# FedLitter — P4 Sprint 2 no Colab
Execute depois de enviar as alteracoes desta branch ao GitHub. Selecione uma GPU T4 em Runtime > Change runtime type. Os resultados sao persistidos no Google Drive para sobreviver a desconexoes.

In [ ]:
!nvidia-smi
BRANCH = 'rodrigo'  # altere para main se a branch ja tiver sido mesclada
!git clone --branch {BRANCH} https://github.com/Guilhermeffda/Servidor-Federated-Learning.git
%cd Servidor-Federated-Learning
!pip install -r requirements-baseline.txt
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
persist = Path('/content/drive/MyDrive/FedLitter-P4')
persist.mkdir(parents=True, exist_ok=True)
!mkdir -p /content/drive/MyDrive/FedLitter-P4/baseline_centralized
!mkdir -p /content/drive/MyDrive/FedLitter-P4/baseline_taco
!mkdir -p /content/drive/MyDrive/FedLitter-P4/centralized_runs
!mkdir -p results runs
!rm -rf results/baseline_centralized results/baseline_taco runs/centralized_baseline
!ln -s /content/drive/MyDrive/FedLitter-P4/baseline_centralized results/baseline_centralized
!ln -s /content/drive/MyDrive/FedLitter-P4/baseline_taco results/baseline_taco
!ln -s /content/drive/MyDrive/FedLitter-P4/centralized_runs runs/centralized_baseline

In [ ]:
!python taco_data/scripts/download_dataset.py
!python taco_data/scripts/regroup_categories.py
!python taco_data/scripts/prepare_experiments.py
!python scripts/train_centralized.py --dry-run

## Baseline centralizada (50 epocas)

In [ ]:
from pathlib import Path
last = Path('runs/centralized_baseline/yolov8n_taco10/weights/last.pt')
if last.exists():
    !python scripts/train_centralized.py --resume {last} --device 0
else:
    !python scripts/train_centralized.py --epochs 50 --imgsz 640 --batch 16 --device 0

## Seis baselines FedAvg TACO-10

In [ ]:
# Pode executar novamente apos uma desconexao: runs com status complete sao puladas.
!python run_config.py --config configs/baseline_taco.yaml --all
!python scripts/validate_baseline.py --results results/baseline_taco

In [ ]:
from google.colab import files
!zip -r p4_sprint2_results.zip /content/drive/MyDrive/FedLitter-P4/baseline_centralized /content/drive/MyDrive/FedLitter-P4/baseline_taco
files.download('p4_sprint2_results.zip')